In [18]:
# %pip install sentence_transformers

In [19]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import TfidfModel
from gensim.corpora import Dictionary, MmCorpus
from sklearn.cluster import KMeans

In [20]:
csv_file = Path("../data/customer_support_tickets_preprocessed.csv")
df = pd.read_csv(csv_file)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket Description charlength,Ticket Description wordlength,Tfidf_ticket_description,Embeddings_ticket_description
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nYour billing zip code is: 71701.\r\n\r\nWe appreciate that you have requested a website address.\r\n\r\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",...,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,290,43,billing code appreciate requested website address double check email address troubleshooting manual,"i'm having an issue with the . please assist. your billing zip code is: 71701. we appreciate that you have requested a website address. please double check your email address. i've tried troubleshooting steps mentioned in the user manual, but the issue persists."
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf you need to change an existing product.\r\n\r\nI'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",...,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,288,44,existing intermittent unexpectedly,"i'm having an issue with the . please assist. if you need to change an existing product. i'm having an issue with the . please assist. if the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly."
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\r\n\r\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly.",...,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,277,42,turning yesterday respond original charger charging,"i'm facing a problem with my . the is not turning on. it was working fine until yesterday, but now it doesn't respond. 1.8.3 i really i'm using the original charger that came with my , but it's not charging properly."
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.",...,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,264,41,interested love happen check feedback multiple,"i'm having an issue with the . please assist. if you have a problem you're interested in and i'd love to see this happen, please check out the feedback. i've already contacted customer support multiple times, but the issue remains unresolved."
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchased}. Please assist.\r\n\r\n\r\nNote: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my {product_purchased}. It used to last much l

### TF-IDF - sklearn

In [21]:
pd.set_option('display.max_colwidth', None)

In [22]:
# Initialise TF-IDF vectorizer to convert text into weighted numerical features
tfidf_vectorizer = TfidfVectorizer(max_df = 0.70, min_df = 5, ngram_range = (1,3))

# Fit the vectorizer on text and transform it into TF-IDF matrix
X_tfidf = tfidf_vectorizer.fit_transform(df['Tfidf_ticket_description'])

In [23]:
X_tfidf.shape

(8044, 1855)

In [24]:
# save the TF-IDF feature matrix in pickle format

joblib.dump( X_tfidf, "../data/tfidf_sklearn.pkl")

['../data/tfidf_sklearn.pkl']

### TF-IDF - genism

In [25]:
# Load the dictionary and bag of words corpus

dictionary = Dictionary.load("../data/gensim_lda_dictionary.dict")
bow_corpus = MmCorpus("../data/gensim_lda_bow_corpus.mm")

In [26]:
# Initialise TF-IDF model using BOW corpus

tfidf_model = TfidfModel(bow_corpus)

# Transform each text in BOW corpus into its TD-IDF representation
tfidf_corpus = [tfidf_model[doc] for doc in bow_corpus]
tfidf_corpus

[[(0, np.float64(0.42565620892036116)),
  (1, np.float64(0.3616888461253772)),
  (2, np.float64(0.3557037902987605)),
  (3, np.float64(0.23426951900130763)),
  (4, np.float64(0.2409868772213425)),
  (5, np.float64(0.4720704810945759)),
  (6, np.float64(0.19608327663428146)),
  (7, np.float64(0.17501100574531558)),
  (8, np.float64(0.3088129672738028)),
  (9, np.float64(0.1488075521549868)),
  (10, np.float64(0.19768338947377168))],
 [(11, np.float64(0.8368154819629819)),
  (12, np.float64(0.38713037154624175)),
  (13, np.float64(0.38713037154624175))],
 [(14, np.float64(0.3951626680510405)),
  (15, np.float64(0.39023350960958164)),
  (16, np.float64(0.37897080131470273)),
  (17, np.float64(0.4201636541765358)),
  (18, np.float64(0.4315608630084287)),
  (19, np.float64(0.4303058571945127))],
 [(3, np.float64(0.3219382623415745)),
  (20, np.float64(0.42991976909030144)),
  (21, np.float64(0.4778423527528013)),
  (22, np.float64(0.5221367357391354)),
  (23, np.float64(0.41273674070132593)

In [27]:
# Save the TF-IDF corpus in Matrix Market (.mm) format using Genism

MmCorpus.serialize("../data/gensim_lda_tfidf_corpus.mm", tfidf_corpus)

### Sentence Embeddings

In [28]:
# Load a pre trained sentence transformer model for generating sentence embeddings

sentence_embedding = SentenceTransformer('all-MiniLM-L6-v2')
X_embed = sentence_embedding.encode(df['Embeddings_ticket_description'].tolist(), show_progress_bar = True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2716.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 252/252 [05:10<00:00,  1.23s/it]


In [29]:
X_embed.shape

(8044, 384)

In [30]:
# Save the generated sentence embeddings to a .npy file

np.save('../data/sentence_embeddings.npy', X_embed)